# Climate Encyclopedia RAG Pipeline

**Purpose:** Run the climate encyclopedia chatbot: load corpus → chunk → embed → retrieve → (optional LLM) → answer.

**Requirements:**
- Run from the **encyclopedia** project root (where the `encyclopedia` package lives).
- For vector search: `pip install sentence-transformers chromadb`
- For LLM answers: run **Ollama** locally (e.g. `ollama run llama3.2:3b`) and set `OLLAMA_MODEL=llama3.2:3b`, or set `OPENAI_API_KEY` for an OpenAI-compatible API.

**Corpus:** Uses aggregated JSON (`temp/chatbot/climate_encyclopedia_entries.json`) if present; otherwise fixture or a tiny in-memory sample. To generate the JSON: `python scripts/count_climate_encyclopedia_entries.py --export`.

## 1. Setup: project root and imports

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on path (run notebook from repo root, or from docs/tutorials)
ROOT = Path.cwd().resolve()
if not (ROOT / "encyclopedia").exists() and (ROOT.parent / "encyclopedia").exists():
    ROOT = ROOT.parent
if not (ROOT / "encyclopedia").exists():
    raise SystemExit("Run this notebook from the encyclopedia repo root (where the 'encyclopedia' package is).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from encyclopedia.chatbot.corpus import load_entries_from_encyclopedia, entries_to_sections
from encyclopedia.chatbot.chunker import chunk_sections
from encyclopedia.chatbot.retrieval import InMemoryRetriever, VectorRetriever
from encyclopedia.chatbot.pipeline import answer_question
from encyclopedia.utils.resources import Resources

print(f"Project root: {ROOT}")

## 2. Load corpus

Finds the first available: `CHATBOT_ENCYCLOPEDIA` env, then `temp/chatbot/climate_encyclopedia_entries.json`, then fixture cache, then root HTML files. If none exist, uses a tiny in-memory sample.

In [ ]:
import os

def find_encyclopedia_path():
    env_path = os.environ.get("CHATBOT_ENCYCLOPEDIA")
    if env_path:
        p = Path(env_path)
        if p.exists():
            return p
    json_path = Path(Resources.TEMP_DIR, "chatbot", "climate_encyclopedia_entries.json")
    if json_path.exists():
        return json_path
    cache_dir = Path(ROOT, "test", "encyclopedia", "fixtures", "cache")
    if cache_dir.exists():
        for p in sorted(cache_dir.glob("encyclopedia_*.html")):
            return p
    for name in ["encyclopedia_output.html", "my_encyclopedia.html", "demo_encyclopedia.html"]:
        p = Path(ROOT, name)
        if p.exists():
            return p
    return None

enc_path = find_encyclopedia_path()
if enc_path:
    print(f"Loading: {enc_path}")
    entries = load_entries_from_encyclopedia(enc_path)
else:
    print("No file found; using minimal in-memory corpus.")
    from encyclopedia.core.encyclopedia import AmiEncyclopedia
    enc = AmiEncyclopedia(title="Example")
    enc.entries = [
        {"term": "climate change", "description_html": "<p>Long-term shifts in temperatures and weather patterns.</p>", "wikidata_id": "Q7942"},
        {"term": "greenhouse gas", "description_html": "<p>Gases that trap heat in the atmosphere.</p>", "wikidata_id": "Q131784"},
        {"term": "carbon dioxide", "description_html": "<p>CO2 is a greenhouse gas produced by burning fossil fuels.</p>", "wikidata_id": "Q1218"},
    ]
    entries = load_entries_from_encyclopedia(enc)

sections = entries_to_sections(entries)
chunks = chunk_sections(sections)
print(f"Entries: {len(entries)}, Sections: {len(sections)}, Chunks: {len(chunks)}")

## 3. Build retriever (keyword + vector)

**InMemoryRetriever** works with no extra installs. **VectorRetriever** needs `sentence-transformers` and `chromadb`. If those are missing, the next cell will skip vector search and you can still use the in-memory retriever for the pipeline.

In [ ]:
# Keyword retriever (no extra deps)
in_memory = InMemoryRetriever(chunks)
for q in ["climate change", "greenhouse gas"]:
    results = in_memory.search(q, k=2)
    print(f"Q: {q}")
    for ch, score in results:
        print(f"  [{score:.2f}] {ch.get('term', '')}: {ch.get('text', '')[:60]}...")
    print()

# Vector retriever (needs: pip install sentence-transformers chromadb)
vector_retriever = None
try:
    persist_dir = Path(Resources.get_temp_dir("tutorials", "climate_rag"), "chroma")
    persist_dir.mkdir(parents=True, exist_ok=True)
    vector_retriever = VectorRetriever(
        chunks,
        persist_directory=str(persist_dir),
        collection_name="climate_rag_notebook",
    )
    print("VectorRetriever ready.")
except ImportError as e:
    print(f"VectorRetriever skipped (install sentence-transformers chromadb): {e}")
    print("Using InMemoryRetriever for the pipeline below.")

retriever = vector_retriever if vector_retriever is not None else in_memory

## 4. Optional LLM

Set `OLLAMA_MODEL` (e.g. `llama3.2:3b`) and run Ollama locally, or set `OPENAI_API_KEY`. If neither is set, the pipeline returns a placeholder message instead of calling an LLM.

In [ ]:
def make_llm_generator():
    ollama_model = os.environ.get("OLLAMA_MODEL")
    if ollama_model:
        try:
            from encyclopedia.chatbot.llm import make_ollama_generator
            return make_ollama_generator(model=ollama_model)
        except Exception:
            return None
    api_key = os.environ.get("OPENAI_API_KEY")
    if api_key:
        try:
            from encyclopedia.chatbot.llm import make_openai_generator
            return make_openai_generator(model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"), api_key=api_key)
        except Exception:
            return None
    return None

llm_generate = make_llm_generator()
if llm_generate is None:
    print("No LLM configured. Set OLLAMA_MODEL or OPENAI_API_KEY to get real answers.")
else:
    print("LLM configured (Ollama or OpenAI-compatible).")

## 5. Run the pipeline: ask a question

The pipeline: **question → retrieve → guardrail (refuse if no/low results) → prompt → LLM → answer**.

In [ ]:
question = "What is climate change?"
out = answer_question(question, retriever, min_score=0.2, llm_generate=llm_generate)

print(f"Question: {question}")
print(f"Refused: {out['refused']}")
print(f"Answer: {out['answer']}")
print(f"Citations: {len(out['citations'])} chunks")

## 6. Try another question

Change the question and run again. If the retriever finds no relevant chunks (or score below threshold), the answer will be the refusal message.

In [ ]:
question = "What are greenhouse gases?"  # Change this and re-run
out = answer_question(question, retriever, min_score=0.2, llm_generate=llm_generate)

print(f"Question: {question}")
print(f"Refused: {out['refused']}")
print(f"Answer: {out['answer']}")
if out.get("citations"):
    for i, c in enumerate(out["citations"][:3]):
        print(f"  Citation {i+1}: [{c.get('section_label')}] {c.get('term', '')}: {c.get('text', '')[:80]}...")